# kaggle_arabart_finetune.ipynb (نسخه اصلاح‌شده)
# هر بخش # ══ CELL X ══ یک سلول جداگانه در notebook است
#
# تغییرات نسبت به نسخه قبلی (با دلیل مستند):
# 1) CELL 2: حذف camel-tools/pyarabic — این دو پکیج علت اثبات‌شده‌ی
#    downgrade شدن numpy از 2.x به 1.26.4 و کرش ValueError در Cell 4 بودند.
#    نصب اکنون فقط برای پکیج‌های واقعاً غایب و با --no-deps انجام می‌شود.
# 2) CELL 3: نام Dataset دیگر hardcode نیست؛ مسیر واقعی حاوی train.jsonl
#    به‌صورت خودکار در سراسر /kaggle/input جستجو می‌شود.
# 3) CELL 8B (جدید): سازگاری خودکار با نام آرگومان eval_strategy/
#    evaluation_strategy بسته به نسخه واقعی transformers نصب‌شده.
# 4) CELL 9: محافظت با min(n, len(dataset)) در برابر دیتاست کوچک.
#
# ⚠️ قبل از اجرا: Run → Restart Session (چون numpy در Session قبلی
# خراب شده و صرف اجرای مجدد سلول‌ها آن را اصلاح نمی‌کند).


In [ ]:
# ══ CELL 1: بررسی GPU + فعال‌سازی CUDA synchronous debugging ══
#
# چرا اضافه شد: خطای قبلی "unspecified launch failure" asynchronous
# است، یعنی traceback واقعی گمراه‌کننده بود (خودِ پیام خطا هم این را
# تأیید کرد). CUDA_LAUNCH_BLOCKING=1 باعث می‌شود هر عملیات GPU بلافاصله
# و همگام گزارش شود تا خط دقیق مقصر پیدا شود. این env varها باید قبل
# از هر فراخوانی CUDA (حتی torch.cuda.is_available در همین سلول)
# تنظیم شوند.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

import torch, sys, json, time, re
from pathlib import Path

print("=" * 50)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap = torch.cuda.get_device_capability()
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")
    print(f"Capability: {cap}")
    print(f"CUDA driver version (torch view): {torch.version.cuda}")
else:
    print("⚠️  CPU only")
print(f"USE_TF={os.environ.get('USE_TF')} | USE_FLAX={os.environ.get('USE_FLAX')}")
print(f"CUDA_LAUNCH_BLOCKING={os.environ.get('CUDA_LAUNCH_BLOCKING')}")
print("=" * 50)

In [ ]:
# ══ CELL 2: نصب وابستگی‌ها (اصلاح نهایی) ═══════════════════════
#
# درس گرفته‌شده از دو باگ قبلی:
# باگ ۱ (numpy ABI): علت camel-tools/pyarabic بود (نیاز به numpy<2.0).
#   → این دو پکیج برای این Notebook اصلاً لازم نیستند (پیش‌پردازش
#     در Phase 1 محلی انجام شده) و برای همیشه حذف می‌شوند.
# باگ ۲ (Tokenizer TypeError): علت حذف کامل version pinning بود.
#   transformers جدیدتر Kaggle، تغییری در پیاده‌سازی توکنایزرهای
#   خانواده Barthez/CamemBERT (که AraBART از آن استفاده می‌کند) دارد
#   که با نسخه tokenizers نصب‌شده ناسازگار است.
#   → راه‌حل: بازگرداندن دقیق همان ترکیب نسخه‌ای که در فایل اصلی
#     پروژه (قبل از دستکاری من) از قبل انتخاب و احتمالاً تست شده بود:
#     transformers==4.41.2 (نسخه‌ای که به این refactor مشکل‌دار نرسیده).
#
# نکته حیاتی: این پکیج‌ها (برخلاف camel-tools) هیچ وابستگی سختی به
# numpy<2.0 ندارند، پس نصب آن‌ها نباید numpy را دوباره خراب کند.
# با این حال، بعد از نصب، numpy را دوباره verify می‌کنیم تا مطمئن شویم.

import importlib
import subprocess
import sys

import numpy
import torch

print("نسخه‌ها قبل از نصب:")
print(f"  numpy        : {numpy.__version__}")
print(f"  torch        : {torch.__version__}")

if not numpy.__version__.startswith("2."):
    raise RuntimeError(
        "numpy در حال حاضر نسخه 2.x نیست! Session قبلی آلوده مانده.\n"
        "همین الان: Run → Restart Session، سپس از Cell 1 دوباره شروع کنید."
    )

# ── ترکیب نسخه‌ی تست‌شده (بدون camel-tools / pyarabic) ─────────
PINNED_PACKAGES = [
    "transformers==4.41.2",
    "tokenizers==0.19.1",   # نسخه‌ی سازگار با transformers==4.41.2
    "peft==0.11.1",
    "accelerate==0.31.0",
    "rouge-score==0.1.2",
    "bert-score==0.3.13",
    "sentencepiece==0.2.0",
    "protobuf==4.25.3",
]
# توجه: datasets عمداً pin نمی‌شود — نسخه‌ی پیش‌فرض Kaggle در Cell 4
# قبلی به‌درستی train/validation/test.jsonl را بارگذاری کرد، پس
# دلیلی برای ریسک کردن با pin کردن آن (که ممکن است numpy<2.0 بخواهد) نیست.

print(f"\nنصب نسخه‌های pin‌شده: {PINNED_PACKAGES}")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + PINNED_PACKAGES
)

print("\nنسخه‌ها بعد از نصب:")
importlib.reload(numpy)
print(f"  numpy        : {numpy.__version__}")

import transformers
import tokenizers as _tok
importlib.reload(transformers)
print(f"  transformers : {transformers.__version__}")
print(f"  tokenizers   : {_tok.__version__}")

assert numpy.__version__.startswith("2."), (
    "⚠️ numpy بعد از نصب مجدداً خراب شد! یکی از پکیج‌های pin‌شده "
    "به‌طور غیرمنتظره numpy<2.0 می‌خواهد. باید بررسی دقیق‌تر شود."
)

# ── تست فوری: آیا خودِ باگ Tokenizer برطرف شد؟ ──────────────────
# به‌جای صبر کردن تا Cell 5 و کرش دیرهنگام، همین‌جا verify می‌کنیم.
print("\nتست بارگذاری AraBART tokenizer (verify باگ برطرف شده) ...")
from transformers import AutoTokenizer

try:
    _test_tok = AutoTokenizer.from_pretrained("moussaKam/AraBART", use_fast=True)
    print("✓ Tokenizer با موفقیت بارگذاری شد (fast).")
    del _test_tok
except TypeError as e:
    print(f"⚠️ همچنان خطا با use_fast=True: {e}")
    print("تلاش با use_fast=False ...")
    _test_tok = AutoTokenizer.from_pretrained("moussaKam/AraBART", use_fast=False)
    print("✓ Tokenizer با use_fast=False بارگذاری شد. Cell 5 باید همین حالت را استفاده کند.")
    del _test_tok

print("\n✓ نصب و verify کامل شد")

In [ ]:
# ══ CELL 3: تنظیمات اصلی + تشخیص خودکار مسیر Dataset ═════════
#
# چرا اصلاح شد:
# نسخه قبلی مسیر دیتاست را با نام ثابت hardcode کرده بود:
#     DATASET_NAME = "arasum-training-data"
#     DATA_DIR = KAGGLE_INPUT / DATASET_NAME
# در اجرای واقعی، DATA_DIR.exists() == False بود، یعنی یا Slug
# دقیق متفاوت است، یا Dataset به این Notebook attach نشده، یا
# ساختار فایل داخل zip تودرتوست.
# راه‌حل: به‌جای فرض کردن نام، مسیر واقعی حاوی train.jsonl را در
# کل /kaggle/input جستجو می‌کنیم (مستقل از نام Dataset یا ساختار zip).

from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

OUTPUT_DIR = KAGGLE_WORKING / "models" / "arabart"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR = KAGGLE_WORKING / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ── تشخیص محتویات /kaggle/input (Diagnostic) ────────────────
print("محتویات /kaggle/input:")
if KAGGLE_INPUT.exists():
    for item in sorted(KAGGLE_INPUT.iterdir()):
        print(f"  📁 {item.name}")
else:
    raise FileNotFoundError(
        "/kaggle/input وجود ندارد. هیچ Dataset‌ای به این Notebook "
        "attach نشده است. از پنل سمت راست → Add Data استفاده کنید."
    )


def find_dataset_dir(kaggle_input: Path) -> Path:
    """
    پوشه‌ای که واقعاً train.jsonl در آن (یا در هر عمق زیرپوشه) قرار دارد
    را پیدا می‌کند. این تابع به نام دقیق Dataset در Kaggle یا به تخت/
    تودرتو بودن ساختار zip وابسته نیست.
    """
    candidates = list(kaggle_input.rglob("train.jsonl"))
    if not candidates:
        all_files = list(kaggle_input.rglob("*"))
        raise FileNotFoundError(
            f"هیچ train.jsonl زیر {kaggle_input} پیدا نشد.\n"
            f"تعداد کل فایل‌های پیدا‌شده: {len(all_files)}\n"
            f"چند نمونه: {[str(f) for f in all_files[:20]]}\n\n"
            "بررسی کنید:\n"
            "  1) Dataset را در پنل 'Add Data' این Notebook attach کرده‌اید؟\n"
            "  2) نام فایل‌ها دقیقاً train.jsonl / validation.jsonl / test.jsonl است؟"
        )
    if len(candidates) > 1:
        print(f"⚠️ چند train.jsonl پیدا شد ({len(candidates)} مورد):")
        for c in candidates:
            print(f"    {c}")
        print(f"  از اولین مورد استفاده می‌شود: {candidates[0]}")

    resolved_dir = candidates[0].parent
    return resolved_dir


DATA_DIR = find_dataset_dir(KAGGLE_INPUT)
print(f"\n✓ Dataset پیدا شد در: {DATA_DIR}")

print(f"\nمحتویات {DATA_DIR}:")
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")

# ── تنظیمات مدل ─────────────────────────────────────────────
MODEL_ID = "moussaKam/AraBART"
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 128
SEED = 42

# ── تشخیص خودکار GPU config ─────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap = torch.cuda.get_device_capability()[0]

    if cap >= 8:        # Ampere+ (A100, RTX 30/40 series)
        USE_FP16, USE_BF16 = False, True
    else:               # قدیمی‌تر (T4، P100، V100)
        USE_FP16, USE_BF16 = True, False

    if vram >= 16:
        BATCH_SIZE, GRAD_ACCUM = 8, 1
    elif vram >= 12:
        BATCH_SIZE, GRAD_ACCUM = 4, 2
    else:
        BATCH_SIZE, GRAD_ACCUM = 2, 4
else:
    USE_FP16, USE_BF16 = False, False
    BATCH_SIZE, GRAD_ACCUM = 2, 4

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM

print(f"\nConfig:")
print(f"  Device: {DEVICE}")
print(f"  FP16: {USE_FP16} | BF16: {USE_BF16}")
print(f"  Batch: {BATCH_SIZE} × {GRAD_ACCUM} = {EFFECTIVE_BATCH}")


In [ ]:
# ══ CELL 4: بارگذاری دیتاست ══════════════════════════════════
# (DATA_DIR در Cell 3 به‌صورت خودکار تشخیص داده شده است)

import json


def load_jsonl(path: Path, max_samples=None):
    records = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            line = line.strip()
            if line:
                try:
                    rec = json.loads(line)
                    if rec.get("text") and rec.get("summary"):
                        records.append(rec)
                except json.JSONDecodeError:
                    pass
    return records


def load_splits(data_dir: Path):
    from datasets import Dataset

    splits = {}
    for split in ["train", "validation", "test"]:
        path = data_dir / f"{split}.jsonl"
        if path.exists():
            recs = load_jsonl(path)
            splits[split] = Dataset.from_list(recs)
            size_mb = path.stat().st_size / 1e6
            print(f"✓ {split}: {len(splits[split]):,} نمونه ({size_mb:.1f} MB)")
        else:
            print(f"⚠️  {split}: پیدا نشد در {path}")

    return splits


print("بارگذاری دیتاست ...")
splits = load_splits(DATA_DIR)

if "train" not in splits:
    raise FileNotFoundError(
        f"train.jsonl در {DATA_DIR} بارگذاری نشد (فایل بود ولی رکورد معتبر نداشت؟).\n"
        "بررسی کنید فیلدهای 'text' و 'summary' در رکوردها موجود باشند."
    )
if "validation" not in splits:
    raise FileNotFoundError(
        f"validation.jsonl در {DATA_DIR} پیدا نشد. برای eval در حین training لازم است."
    )

print("\nنمونه‌های اول:")
for i in range(min(2, len(splits["train"]))):
    ex = splits["train"][i]
    print(f"  text ({len(ex['text'].split())} کلمه): {ex['text'][:80]}...")
    print(f"  summary ({len(ex['summary'].split())} کلمه): {ex['summary'][:60]}...")
    print()


In [ ]:
# ══ CELL 5: tokenizer و مدل (با fallback خودکار) ══════════════
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import set_seed

set_seed(SEED)

print(f"بارگذاری tokenizer: {MODEL_ID}")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    print("✓ Fast tokenizer با موفقیت بارگذاری شد.")
except TypeError as e:
    # همان باگ شناخته‌شده‌ی Unigram/vocab در توکنایزرهای خانواده
    # Barthez/CamemBERT با نسخه‌های ناسازگار transformers/tokenizers.
    print(f"⚠️ Fast tokenizer شکست خورد ({e})")
    print("→ Fallback به use_fast=False ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
    print("✓ Slow tokenizer با موفقیت بارگذاری شد.")

print(f"بارگذاری مدل: {MODEL_ID}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.to(DEVICE)

params = sum(p.numel() for p in model.parameters())
print(f"✓ مدل بارگذاری شد: {params:,} پارامتر")
print(f"  Vocab size: {tokenizer.vocab_size:,}")
print(f"  Pad token ID: {tokenizer.pad_token_id}")
print(f"  Tokenizer type: {type(tokenizer).__name__}")
print(f"  Is fast: {tokenizer.is_fast}")

In [ ]:
# ══ CELL 6: Tokenization ══════════════════════════════════════

pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

def tokenize_fn(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False,
    )
    label_enc = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = [
        [t if t != pad_id else -100 for t in ids]
        for ids in label_enc["input_ids"]
    ]
    return model_inputs


print("Tokenize کردن ...")
tokenized = {}
for split_name, ds in splits.items():
    tokenized[split_name] = ds.map(
        tokenize_fn, batched=True,
        remove_columns=ds.column_names,
        desc=f"Tokenize {split_name}",
    )
    print(f"✓ {split_name}: {len(tokenized[split_name]):,} نمونه")

# نمایش نمونه
ex = tokenized["train"][0]
labels_real = [l for l in ex["labels"] if l != -100]
print(f"\nنمونه اول:")
print(f"  input_ids: {len(ex['input_ids'])} tokens")
print(f"  labels (real): {len(labels_real)} tokens")
print(f"  input: {tokenizer.decode(ex['input_ids'][:30], skip_special_tokens=True)[:80]}...")
print(f"  label: {tokenizer.decode(labels_real[:20], skip_special_tokens=True)[:60]}...")

# اندازه‌گیری truncation
for split_name, ds in tokenized.items():
    trunc = sum(1 for ex in ds if len(ex["input_ids"]) >= MAX_SOURCE_LENGTH)
    total = len(ds)
    print(f"truncation [{split_name}]: {trunc}/{total} ({100*trunc/total:.1f}%)")


In [ ]:
# ══ CELL 7: LoRA Setup (اصلاح‌شده) ════════════════════════════
from peft import LoraConfig, get_peft_model, TaskType

found_modules = set()
for name, _ in model.named_modules():
    for c in ["q_proj", "v_proj", "k_proj", "out_proj"]:
        if name.endswith(c):
            found_modules.add(c)

TARGET_MODULES = sorted(list(found_modules)) or ["q_proj", "v_proj"]
print(f"Target modules: {TARGET_MODULES}")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, lora_config)

# ── فیکس حیاتی: لازم برای سازگاری LoRA + gradient_checkpointing ──
# بدون این خط، چون تمام پارامترهای مدل پایه (embedding شامل) در LoRA
# freeze می‌شوند، gradient_checkpointing نمی‌تواند backward graph
# معتبری بسازد. این ناسازگاری بسته به نسخه torch/CUDA گاهی خطای
# پایتونی واضح می‌دهد و گاهی (مثل اینجا) مستقیم CUDA launch failure.
model.enable_input_require_grads()
print("✓ enable_input_require_grads() فعال شد (سازگاری با gradient_checkpointing)")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ LoRA: {trainable:,} / {total:,} قابل‌آموزش ({100*trainable/total:.4f}%)")

if trainable == 0:
    raise RuntimeError("هیچ پارامتر قابل‌آموزشی! target_modules اشتباه است.")

In [ ]:
# ══ CELL 8: compute_metrics ══════════════════════════════════

import numpy as np
from rouge_score import rouge_scorer as rs

class ArabicTok:
    def tokenize(self, text):
        text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)
        return [t for t in text.split() if t]

rouge_scorer_obj = rs.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,
    tokenizer=ArabicTok(),
)

vocab_size = tokenizer.vocab_size
_pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # کلیپ کردن برای جلوگیری از OverflowError
    predictions = np.clip(
        predictions.astype(np.int64), 0, vocab_size - 1
    ).astype(np.int32)
    labels_fixed = np.where(labels != -100, labels, _pad_id)
    labels_fixed = np.clip(
        labels_fixed.astype(np.int64), 0, vocab_size - 1
    ).astype(np.int32)

    decoded_preds, decoded_refs = [], []
    for pred_ids, label_ids in zip(predictions, labels_fixed):
        try:
            p = tokenizer.decode(
                pred_ids.tolist(), skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            ).strip()
        except Exception:
            p = ""
        try:
            r = tokenizer.decode(
                label_ids.tolist(), skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            ).strip()
        except Exception:
            r = ""
        decoded_preds.append(p)
        decoded_refs.append(r)

    r1, r2, rL = [], [], []
    for p, r in zip(decoded_preds, decoded_refs):
        if p and r:
            try:
                s = rouge_scorer_obj.score(r, p)
                r1.append(s["rouge1"].fmeasure)
                r2.append(s["rouge2"].fmeasure)
                rL.append(s["rougeL"].fmeasure)
            except Exception:
                pass

    if not r1:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

    return {
        "rouge1": round(sum(r1) / len(r1), 4),
        "rouge2": round(sum(r2) / len(r2), 4),
        "rougeL": round(sum(rL) / len(rL), 4),
    }

print("✓ compute_metrics آماده")


In [ ]:
# ══ CELL 8B (جدید): سازگاری خودکار نسخه Transformers ══════════
#
# چرا لازم است:
# چون دیگر transformers را force-pin نمی‌کنیم (Cell 2)، نسخه
# واقعی نصب‌شده در این Session نامعلوم است. در نسخه‌های مختلف
# Transformers، نام آرگومان استراتژی ارزیابی بین
# evaluation_strategy (نسخه‌های قدیمی‌تر) و eval_strategy
# (نسخه‌های >=4.46) متفاوت است. اگر نام اشتباه پاس داده شود:
#     TypeError: __init__() got an unexpected keyword argument
# این تابع کمکی، نام صحیح را از روی signature واقعی نصب‌شده
# تشخیص می‌دهد — نه از روی حدس یا نسخه مستندشده در کاغذ.

import inspect
from transformers import Seq2SeqTrainingArguments

_TA_PARAMS = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
_EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"

print(f"نسخه transformers نصب‌شده: {transformers.__version__}")
print(f"نام آرگومان استراتژی ارزیابی در این نسخه: {_EVAL_STRATEGY_KEY}")


def build_training_args(**kwargs) -> Seq2SeqTrainingArguments:
    """
    Seq2SeqTrainingArguments را با نام صحیح آرگومان
    eval_strategy/evaluation_strategy بر اساس نسخه نصب‌شده می‌سازد.
    فراخوانی همیشه از کلید 'eval_strategy' استفاده می‌کند؛
    این تابع در صورت نیاز آن را به نام قدیمی تبدیل می‌کند.
    """
    if "eval_strategy" in kwargs:
        value = kwargs.pop("eval_strategy")
        kwargs[_EVAL_STRATEGY_KEY] = value
    return Seq2SeqTrainingArguments(**kwargs)


print("✓ build_training_args آماده")


In [ ]:
# ══ CELL 9-RAW: تست خام مینیمال بدون Trainer/Accelerate ═══════
#
# هدف: Cell 9-DIAGNOSTIC نشان داد gradient_checkpointing مقصر نیست
# (همان خطا در training_step->empty_cache() تکرار شد). چون خطاهای
# CUDA asynchronous هستند، اینجا با حذف کامل لایه Trainer/Accelerate
# و اجرای مستقیم PyTorch + CUDA_LAUNCH_BLOCKING=1، دقیقاً مشخص می‌کنیم
# کدام مرحله (FP32 forward / FP32 backward / FP16 autocast) خراب است.

import torch
from transformers import DataCollatorForSeq2Seq

print(f"CUDA_LAUNCH_BLOCKING={os.environ.get('CUDA_LAUNCH_BLOCKING')}")
print(f"TORCH_USE_CUDA_DSA={os.environ.get('TORCH_USE_CUDA_DSA')}")

model.train()
model.to(DEVICE)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    padding=True, pad_to_multiple_of=8,
    label_pad_token_id=-100,
)
raw_examples = [tokenized["train"][i] for i in range(2)]
batch = collator(raw_examples)
batch = {k: v.to(DEVICE) for k, v in batch.items()}

print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {tuple(v.shape)} dtype={v.dtype}")

# ── تست ۱: Forward با FP32 (بدون هیچ autocast) ──────────────────
print("\n--- تست ۱: FP32 forward (inference-only) ---")
try:
    with torch.no_grad():
        out = model(**batch)
    print(f"✓ تست ۱ موفق. Loss: {out.loss.item():.4f}")
except RuntimeError as e:
    print(f"❌ تست ۱ شکست خورد:\n{e}")
    raise

# ── تست ۲: Forward + Backward با FP32 (بدون autocast) ───────────
print("\n--- تست ۲: FP32 forward + backward ---")
try:
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=5e-5
    )
    optimizer.zero_grad()
    out = model(**batch)
    out.loss.backward()
    optimizer.step()
    print(f"✓ تست ۲ موفق. Loss: {out.loss.item():.4f}")
except RuntimeError as e:
    print(f"❌ تست ۲ شکست خورد:\n{e}")
    raise

# ── تست ۳: Forward + Backward با FP16 autocast (شبیه training واقعی) ──
print("\n--- تست ۳: FP16 autocast forward + backward (مثل training واقعی) ---")
try:
    scaler = torch.amp.GradScaler("cuda")
    optimizer.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        out = model(**batch)
    scaler.scale(out.loss).backward()
    scaler.step(optimizer)
    scaler.update()
    print(f"✓ تست ۳ موفق. Loss: {out.loss.item():.4f}")
except RuntimeError as e:
    print(f"❌ تست ۳ شکست خورد:\n{e}")
    raise

print("\n✓✓✓ هر سه تست موفق شدند → مشکل احتمالاً مختص لایه Trainer/Accelerate است، نه خودِ مدل/CUDA")

In [ ]:
# ══ CELL 9: Smoke Test (اجباری قبل از training کامل) ════════
#
# اصلاح: استفاده از min(n, len(dataset)) به‌جای عدد ثابت،
# تا اگر train/validation کوچک‌تر از حد انتظار بود کرش نکند.
# اصلاح: استفاده از build_training_args به‌جای فراخوانی مستقیم
# Seq2SeqTrainingArguments (رفع ریسک TypeError مربوط به
# eval_strategy/evaluation_strategy).

from transformers import (
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

print("اجرای Smoke Test (3 step) ...")

smoke_args = build_training_args(
    output_dir=str(OUTPUT_DIR / "smoke"),
    max_steps=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="steps",
    eval_steps=3,
    save_strategy="steps",
    save_steps=3,
    predict_with_generate=True,
    generation_max_length=32,
    generation_num_beams=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # ← فیکس اضافه
    logging_steps=1,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    seed=SEED,
)

smoke_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    padding=True, pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

smoke_train = tokenized["train"].select(range(min(8, len(tokenized["train"]))))
smoke_eval = tokenized["validation"].select(range(min(4, len(tokenized["validation"]))))

smoke_trainer = Seq2SeqTrainer(
    model=model, args=smoke_args,
    train_dataset=smoke_train,
    eval_dataset=smoke_eval,
    tokenizer=tokenizer,
    data_collator=smoke_collator,
    compute_metrics=compute_metrics,
)

smoke_result = smoke_trainer.train()

assert smoke_result.training_loss == smoke_result.training_loss, "❌ Loss NaN در Smoke Test!"

print(f"✓ Smoke Test Loss: {smoke_result.training_loss:.4f}")
print("✓ Smoke Test موفق! ادامه به Cell بعدی")


In [ ]:
# ══ CELL 10: Training کامل ═══════════════════════════════════
# اصلاح: استفاده از build_training_args به‌جای فراخوانی مستقیم
# Seq2SeqTrainingArguments.

from transformers import EarlyStoppingCallback

# بررسی checkpoint برای resume
RESUME_FROM = None
existing_checkpoints = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda d: int(d.name.split("-")[-1]),
    reverse=True,
)
if existing_checkpoints:
    RESUME_FROM = str(existing_checkpoints[0])
    print(f"Resume از checkpoint: {RESUME_FROM}")
else:
    print("شروع از ابتدا")

training_args = build_training_args(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    seed=SEED,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    save_total_limit=3,
    logging_steps=100,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    padding=True, pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"شروع training: {len(tokenized['train']):,} نمونه | {BATCH_SIZE}×{GRAD_ACCUM}={EFFECTIVE_BATCH} batch")
t0 = time.time()
result = trainer.train(resume_from_checkpoint=RESUME_FROM)
duration = time.time() - t0

print(f"\n✓ Training کامل شد!")
print(f"  زمان: {duration/3600:.2f} ساعت")
print(f"  Loss: {result.training_loss:.4f}")


In [ ]:
# ══ CELL 11: ذخیره مدل ═══════════════════════════════════════

print("ذخیره مدل ...")

# LoRA adapter
adapter_dir = OUTPUT_DIR / "lora_adapter"
adapter_dir.mkdir(exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"✓ LoRA adapter: {adapter_dir}")

# مدل merge‌شده
try:
    print("Merge کردن ...")
    merged_model = model.merge_and_unload()
    merged_dir = OUTPUT_DIR / "merged"
    merged_dir.mkdir(exist_ok=True)
    merged_model.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    print(f"✓ مدل merge‌شده: {merged_dir}")
except Exception as e:
    print(f"⚠️  merge ناموفق: {e}")

# metadata
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
metadata = {
    "training_time_hours": round(duration/3600, 2),
    "training_loss": result.training_loss,
    "base_model": MODEL_ID,
    "seed": SEED,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "precision": "bf16" if USE_BF16 else ("fp16" if USE_FP16 else "fp32"),
    "transformers_version": transformers.__version__,
    "lora": {
        "r": 16, "alpha": 32, "dropout": 0.1,
        "target_modules": TARGET_MODULES,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_pct": round(trainable/total*100, 4),
    },
    "training": {
        "epochs": 3, "lr": 5e-5,
        "batch": BATCH_SIZE, "accum": GRAD_ACCUM,
        "effective_batch": EFFECTIVE_BATCH,
    },
}

with open(OUTPUT_DIR / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ همه فایل‌ها در {OUTPUT_DIR} ذخیره شدند")

print("\nفایل‌های خروجی:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size / 1e6
        print(f"  {p.relative_to(OUTPUT_DIR)}: {size:.1f} MB")


In [ ]:
# ══ CELL 12: ارزیابی نهایی (اختیاری) ════════════════════════

print("ارزیابی روی validation ...")
eval_results = trainer.evaluate()
print("\nنتایج:")
for k, v in eval_results.items():
    print(f"  {k}: {v}")

# بررسی هدف‌های پروژه
r1 = eval_results.get("eval_rouge1", 0)
r2 = eval_results.get("eval_rouge2", 0)
rL = eval_results.get("eval_rougeL", 0)

print("\n📊 بررسی اهداف پروپوزال (validation، نه test نهایی):")
print(f"  ROUGE-1: {r1:.4f} {'✓' if r1 >= 0.45 else '✗'} (هدف ≥0.45)")
print(f"  ROUGE-2: {r2:.4f} {'✓' if r2 >= 0.25 else '✗'} (هدف ≥0.25)")
print(f"  ROUGE-L: {rL:.4f} {'✓' if rL >= 0.40 else '✗'} (هدف ≥0.40)")
print("\n⚠️ توجه: این ارزیابی روی validation است، نه test نهایی دست‌نخورده.")
print("طبق TASK-203، ارزیابی نهایی باید فقط یک‌بار روی test.jsonl انجام شود.")

# ذخیره گزارش
report = {
    "split": "validation",
    "evaluation": eval_results,
    "targets": {
        "rouge1": {"target": 0.45, "achieved": r1, "pass": r1 >= 0.45},
        "rouge2": {"target": 0.25, "achieved": r2, "pass": r2 >= 0.25},
        "rougeL": {"target": 0.40, "achieved": rL, "pass": rL >= 0.40},
    },
    "metadata": metadata,
}

with open(REPORTS_DIR / "phase2_results_validation.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\n✓ گزارش: {REPORTS_DIR / 'phase2_results_validation.json'}")


## خلاصه اصلاحات این نسخه

1. **numpy ABI crash (Cell 4)** → علت: نصب `camel-tools`/`pyarabic` در Cell 2 باعث downgrade خودکار numpy از 2.x به 1.26.4 شد. این دو پکیج در محیط training لازم نبودند (پیش‌پردازش قبلاً در Phase 1 انجام شده) و حذف شدند.
2. **Dataset path not found (Cell 3/4)** → علت: نام Dataset به‌صورت ثابت فرض شده بود. جایگزین شد با جستجوی خودکار `train.jsonl` در کل `/kaggle/input`.
3. **ریسک TypeError احتمالی روی eval_strategy (Cell 8B/9/10)** → چون دیگر نسخه transformers force-pin نمی‌شود، به‌صورت پیشگیرانه یک لایه سازگاری اضافه شد.
4. **ریسک crash در Smoke Test با دیتاست کوچک (Cell 9)** → با `min(n, len(dataset))` رفع شد.

نتیجه مورد انتظار در Cell 12 صرفاً معیار **validation** حین training است، نه ارزیابی نهایی TASK-203 که باید روی `test.jsonl` دست‌نخورده (فقط یک‌بار) انجام شود.
